# keyboard_config

> Shared keyboard navigation configuration for the combined Phase 2 step

In [ ]:
#| default_exp components.keyboard_config

In [ ]:
#| export
from typing import Any, Tuple, Optional

from fasthtml.common import Details, Summary, Div, Script

# DaisyUI components
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui
from cjm_fasthtml_daisyui.components.data_display.collapse import (
    collapse, collapse_title, collapse_content, collapse_modifiers
)

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.core.base import combine_classes

# Keyboard navigation library
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager
from cjm_fasthtml_keyboard_navigation.components.hints import render_keyboard_hints
from cjm_fasthtml_keyboard_navigation.components.system import render_keyboard_system

# Card stack library
from cjm_fasthtml_card_stack.keyboard.actions import build_card_stack_url_map

# Local HTML IDs
from cjm_fasthtml_workflow_transcript_decomp.core.html_ids import StructureDecompHtmlIds

# Decomposition-specific keyboard config (building blocks)
from cjm_fasthtml_workflow_transcript_decomp.decomposition.components.keyboard_config import (
    SD_DECOMP_ENTER_SPLIT_BTN, SD_DECOMP_EXIT_SPLIT_BTN,
    SD_DECOMP_SPLIT_BTN, SD_DECOMP_MERGE_BTN, SD_DECOMP_UNDO_BTN,
    create_decomp_kb_parts,
)
from cjm_fasthtml_workflow_transcript_decomp.decomposition.components.card_stack_config import (
    DECOMP_CS_CONFIG, DECOMP_CS_IDS, DECOMP_CS_BTN_IDS,
    DECOMP_TS_IDS,
)

# Alignment-specific keyboard config (building blocks)
from cjm_fasthtml_workflow_transcript_decomp.alignment.components.keyboard_config import (
    create_align_kb_parts,
)
from cjm_fasthtml_workflow_transcript_decomp.alignment.components.card_stack_config import (
    ALIGN_CS_CONFIG, ALIGN_CS_IDS, ALIGN_CS_BTN_IDS,
)

# URL bundles
from cjm_fasthtml_workflow_transcript_decomp.decomposition.models import DecompUrls
from cjm_fasthtml_workflow_transcript_decomp.alignment.models import AlignmentUrls

# Debug flag for keyboard system tracing (set False in production)
DEBUG_KB_SYSTEM = True

## Keyboard Hints Component

Generic collapsible keyboard hints renderer. Works with any `ZoneManager`
instance, wrapping the library's `render_keyboard_hints` in a DaisyUI collapse.

In [ ]:
#| export
def render_keyboard_hints_collapsible(
    manager:ZoneManager,  # Keyboard zone manager with actions configured
    container_id:str="sd-keyboard-hints",  # HTML ID for the hints container
    include_zone_switch:bool=False,  # Whether to include zone switch hints
) -> Any:  # Collapsible keyboard hints component
    """Render keyboard shortcut hints in a collapsible DaisyUI collapse."""
    hints = render_keyboard_hints(
        manager,
        include_navigation=True,
        include_zone_switch=include_zone_switch,
        badge_style="outline",
        container_id=container_id,
        use_icons=False
    )

    return Details(
        Summary(
            "Keyboard Shortcuts",
            cls=combine_classes(collapse_title, font_size.sm, font_weight.medium)
        ),
        Div(
            hints,
            cls=collapse_content
        ),
        cls=combine_classes(collapse, collapse_modifiers.arrow, bg_dui.base_200)
    )

## Keyboard System Builder

Builds the combined keyboard system with both decomposition and alignment zones.
Returns `(manager, system)` tuple where manager is needed for hints rendering
and system contains the script/inputs/buttons for the stable KB container.

The combined KB system is always built with both zones, regardless of which
column has initialized first. Zone switching works immediately; zones with
no DOM content yet will simply have no focusable items until content loads.

In [ ]:
#| export
# Zone change callback name — global JS function called when zone switches
ZONE_CHANGE_CALLBACK = "onCombinedZoneChange"


def build_combined_kb_system(
    decomp_urls:DecompUrls,  # URL bundle for decomposition routes
    align_urls:AlignmentUrls,  # URL bundle for alignment routes
) -> Tuple[ZoneManager, Any]:  # (keyboard manager, rendered keyboard system)
    """Build combined keyboard system with decomp and alignment zones."""
    if DEBUG_KB_SYSTEM:
        print("[KB_SYSTEM] build_combined_kb_system called")

    # Get decomp-specific building blocks
    decomp_zone, decomp_actions, decomp_modes = create_decomp_kb_parts(
        ids=DECOMP_CS_IDS,
        button_ids=DECOMP_CS_BTN_IDS,
        config=DECOMP_CS_CONFIG,
    )

    # Get alignment-specific building blocks
    align_zone, align_actions, align_modes = create_align_kb_parts(
        ids=ALIGN_CS_IDS,
        button_ids=ALIGN_CS_BTN_IDS,
        config=ALIGN_CS_CONFIG,
    )

    if DEBUG_KB_SYSTEM:
        print(f"[KB_SYSTEM] decomp_zone.id: {decomp_zone.id}")
        print(f"[KB_SYSTEM] align_zone.id: {align_zone.id}")
        print(f"[KB_SYSTEM] decomp_actions count: {len(decomp_actions)}")
        print(f"[KB_SYSTEM] align_actions count: {len(align_actions)}")

    # Combine modes (only decomp has split mode)
    all_modes = (*decomp_modes, *align_modes)

    # Combine actions from both zones
    all_actions = (*decomp_actions, *align_actions)

    # Assemble into ZoneManager with zone switching enabled
    kb_manager = ZoneManager(
        zones=(decomp_zone, align_zone),
        actions=all_actions,
        modes=all_modes,
        initial_zone_id=decomp_zone.id,
        prev_zone_key="ArrowLeft",
        next_zone_key="ArrowRight",
        on_zone_change=ZONE_CHANGE_CALLBACK,
        state_hidden_inputs=True,
    )

    # Build URL maps for both stacks
    decomp_include_selector = (
        f"#{DECOMP_CS_IDS.focused_index_input}, "
        f"#{DECOMP_TS_IDS.anchor_input}"
    )
    align_include_selector = f"#{ALIGN_CS_IDS.focused_index_input}"

    # Decomp URL mappings
    decomp_url_map = {
        **build_card_stack_url_map(DECOMP_CS_BTN_IDS, decomp_urls.card_stack),
        SD_DECOMP_ENTER_SPLIT_BTN: decomp_urls.enter_split,
        SD_DECOMP_EXIT_SPLIT_BTN: decomp_urls.exit_split,
        SD_DECOMP_SPLIT_BTN: decomp_urls.split,
        SD_DECOMP_MERGE_BTN: decomp_urls.merge,
        SD_DECOMP_UNDO_BTN: decomp_urls.undo,
    }

    # Alignment URL mappings (navigation only — no undo for alignment)
    align_url_map = {
        **build_card_stack_url_map(ALIGN_CS_BTN_IDS, align_urls.card_stack),
    }

    # Combined URL map
    url_map = {**decomp_url_map, **align_url_map}

    # Target maps (each stack targets its own card_stack element)
    decomp_target = f"#{DECOMP_CS_IDS.card_stack}"
    align_target = f"#{ALIGN_CS_IDS.card_stack}"
    target_map = {
        **{btn_id: decomp_target for btn_id in decomp_url_map},
        **{btn_id: align_target for btn_id in align_url_map},
    }

    # Include maps (each stack includes its own focused_index input)
    include_map = {
        **{btn_id: decomp_include_selector for btn_id in decomp_url_map},
        **{btn_id: align_include_selector for btn_id in align_url_map},
    }

    # Swap map (none for all — OOB swaps handle updates)
    swap_map = {btn_id: "none" for btn_id in url_map}

    kb_system = render_keyboard_system(
        kb_manager,
        url_map=url_map,
        target_map=target_map,
        include_map=include_map,
        swap_map=swap_map,
        show_hints=False,
        include_state_inputs=True,
    )

    if DEBUG_KB_SYSTEM:
        print(f"[KB_SYSTEM] combined kb_system built successfully")

    return kb_manager, kb_system

## Zone Change JavaScript

Generates JavaScript for zone change handling including:
- Global callback function that runs when zones switch via keyboard
- Visual styling updates (ring/opacity on columns)
- Hidden input updates for active column state
- Chrome swap trigger via HTMX button click
- Click handlers on columns for mouse-driven zone switching

In [ ]:
#| export
# Hidden button ID for chrome swap HTMX trigger
SWITCH_CHROME_BTN_ID = "sd-switch-chrome-btn"


def generate_zone_change_js(
    switch_chrome_url:str="",  # URL for chrome swap handler (empty = no swap)
) -> Script:  # Script element with zone change callback and click handlers
    """Generate JavaScript for zone change handling and column click handlers."""
    # Zone IDs from card stack configs
    decomp_zone_id = DECOMP_CS_IDS.card_stack
    align_zone_id = ALIGN_CS_IDS.card_stack

    # Card stack prefixes for syncCountDropdown calls
    decomp_prefix = DECOMP_CS_CONFIG.prefix
    align_prefix = ALIGN_CS_CONFIG.prefix

    # Column container IDs
    decomp_col_id = StructureDecompHtmlIds.DECOMP_COLUMN
    align_col_id = StructureDecompHtmlIds.ALIGNMENT_COLUMN
    active_input_id = StructureDecompHtmlIds.ACTIVE_COLUMN_INPUT

    # Chrome swap trigger with dropdown sync (optional)
    chrome_swap_js = ""
    if switch_chrome_url:
        chrome_swap_js = f"""
            // Trigger chrome swap via hidden HTMX button
            const chromeSwitchBtn = document.getElementById('{SWITCH_CHROME_BTN_ID}');
            if (chromeSwitchBtn) {{
                // Add one-time listener to sync dropdown after chrome swap settles
                function onChromeSettle(evt) {{
                    // Sync the active card stack's dropdown from localStorage
                    const activePrefix = (newZoneId === decomp_zone_id) ? '{decomp_prefix}' : '{align_prefix}';
                    if (window.cardStacks?.[activePrefix]?.syncCountDropdown) {{
                        window.cardStacks[activePrefix].syncCountDropdown();
                    }}
                    document.body.removeEventListener('htmx:afterSettle', onChromeSettle);
                }}
                document.body.addEventListener('htmx:afterSettle', onChromeSettle);
                chromeSwitchBtn.click();
            }}
        """

    js_code = f"""
    (function() {{
        const decomp_zone_id = '{decomp_zone_id}';
        const align_zone_id = '{align_zone_id}';
        const decomp_col_id = '{decomp_col_id}';
        const align_col_id = '{align_col_id}';
        const active_input_id = '{active_input_id}';

        function updateColumnStyles(activeZoneId) {{
            const decompCol = document.getElementById(decomp_col_id);
            const alignCol = document.getElementById(align_col_id);
            const activeInput = document.getElementById(active_input_id);

            if (!decompCol || !alignCol) return;

            const isDecompActive = activeZoneId === decomp_zone_id;

            // Ring classes for active indicator
            decompCol.classList.toggle('ring-1', isDecompActive);
            decompCol.classList.toggle('ring-primary', isDecompActive);
            alignCol.classList.toggle('ring-1', !isDecompActive);
            alignCol.classList.toggle('ring-secondary', !isDecompActive);

            // Opacity for inactive column
            decompCol.classList.toggle('opacity-70', !isDecompActive);
            alignCol.classList.toggle('opacity-70', isDecompActive);

            // Update hidden input
            if (activeInput) {{
                activeInput.value = isDecompActive ? 'decomp' : 'align';
            }}
        }}

        // Global callback for ZoneManager (called on keyboard zone switch)
        window.{ZONE_CHANGE_CALLBACK} = function(newZoneId, prevZoneId) {{
            if (window.DEBUG_KB_SYSTEM) {{
                console.log('[KB_SYSTEM] Zone change:', prevZoneId, '->', newZoneId);
            }}
            updateColumnStyles(newZoneId);
            {chrome_swap_js}
        }};

        // Click handler for column focus switching
        function handleColumnClick(targetZoneId) {{
            // Skip if already active
            const state = window.kbNav?.getState?.();
            if (state && state.activeZoneId === targetZoneId) return;

            // Skip if in split mode (don't allow accidental zone switch)
            if (state && state.currentMode === 'split') return;

            // Switch zone using keyboard navigation library API
            if (window.kbNav?.setActiveZone) {{
                window.kbNav.setActiveZone(targetZoneId);
            }}
        }}

        // Set up click handlers on columns
        function setupClickHandlers() {{
            const decompCol = document.getElementById(decomp_col_id);
            const alignCol = document.getElementById(align_col_id);

            if (decompCol) {{
                decompCol.addEventListener('mousedown', function(e) {{
                    handleColumnClick(decomp_zone_id);
                }});
            }}

            if (alignCol) {{
                alignCol.addEventListener('mousedown', function(e) {{
                    handleColumnClick(align_zone_id);
                }});
            }}
        }}

        // Initial update (decomp zone is active by default)
        updateColumnStyles(decomp_zone_id);

        // Set up click handlers
        setupClickHandlers();
    }})();
    """

    return Script(js_code)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()